In [1]:
data_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers"

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import wandb
import os

In [3]:
# =======================
# STEP 0: Initialize wandb
# =======================
wandb.init(project="alexnet-flowers-v2", config={
    "epochs": 10,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "alexnet",
    "pretrained": True,
    "input_size": 128
})

# Shortcut to config values
config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\sangr\_netrc.
wandb: Currently logged in as: sangram01 (sangram01-srm-institute-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "val"),
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [5]:
# ===========================
# STEP 2: Load Pretrained Model
# ===========================

alexnet = models.alexnet(pretrained=config.pretrained)
# alexnet.classifier[6] = nn.Linear(4096, 5)

# Freeze all layers
for param in alexnet.parameters():
    param.requires_grad = False

# Replace the final classifier layer (trainable)
alexnet.classifier[6] = nn.Linear(4096, 5)                   ## we have 5 flowers data so we defined last layer as 5 o/p node

# Only the new final layer should be trainable
for param in alexnet.classifier[6].parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
alexnet = alexnet.to(device)

# Watch the model's weights and gradients
wandb.watch(alexnet, log="all", log_freq=10)

C:\PYTHON_ENV\env_object_detection\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\PYTHON_ENV\env_object_detection\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


cuda


In [6]:
# ===================
# STEP 3: Loss & Optimizer
# ===================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=config.learning_rate)

In [7]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        running_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct = (preds == labels).sum().item()
            train_correct += batch_correct
            train_total += labels.size(0)

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
        print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
        print(f"Validation Accuracy: {val_acc:.4f}")


## In PyTorch many of NN operation & functions are defined in classes

In [8]:
# ===================
# Train the model
# ===================
train_model(alexnet, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/10
------------------------------
[Batch 10/126] Loss: 1.1274, Batch Acc: 0.6562
[Batch 20/126] Loss: 0.4937, Batch Acc: 0.7500
[Batch 30/126] Loss: 0.4949, Batch Acc: 0.8125
[Batch 40/126] Loss: 0.9076, Batch Acc: 0.6875
[Batch 50/126] Loss: 0.5000, Batch Acc: 0.7812
[Batch 60/126] Loss: 0.5104, Batch Acc: 0.8125
[Batch 70/126] Loss: 0.4051, Batch Acc: 0.8438
[Batch 80/126] Loss: 0.5765, Batch Acc: 0.8438
[Batch 90/126] Loss: 0.5562, Batch Acc: 0.7812
[Batch 100/126] Loss: 0.6477, Batch Acc: 0.7812
[Batch 110/126] Loss: 0.5092, Batch Acc: 0.8125
[Batch 120/126] Loss: 0.3532, Batch Acc: 0.8438
Epoch 1 Summary - Loss: 74.9018, Train Accuracy: 0.7792
Validation Accuracy: 0.8556

Epoch 2/10
------------------------------
[Batch 10/126] Loss: 0.4672, Batch Acc: 0.8750
[Batch 20/126] Loss: 0.3073, Batch Acc: 0.8438
[Batch 30/126] Loss: 0.3658, Batch Acc: 0.8125
[Batch 40/126] Loss: 0.3449, Batch Acc: 0.8125
[Batch 50/126] Loss: 0.2669, Batch Acc: 0.9688
[Batch 60/126] Loss: 0.4267,